# Kettlerock engine walkthrough

This notebook runs the Kettlerock data engine step by step and shows what each step produces. The engine itself lives in `src/kettlerock_engine`; the notebook demonstrates and explains it, and the tests in `tests/` verify it independently.

Every business assumption the engine uses is read from `01_calibration/calibrated_parameters.csv` by its parameter ID. Structure that is authored rather than calibrated (the store list, the product catalogue, the closure rules) is kept as readable files in `config/structure`.

## Milestone 1: structure

The structure step builds the tables every later step hangs on: the fiscal calendar, provincial holidays and closures, markets and stores, products and their opening costs, the chart of accounts, the planning versions and the debt facilities. Structure is deterministic: it uses no random draws, so identifiers and attributes are identical in every regeneration.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))      # run from 02_engine/notebooks

import pandas as pd
from kettlerock_engine.config import load_config
from kettlerock_engine.params import Params
from kettlerock_engine import calendar, structure

cfg = load_config()
params = Params(cfg.calibrated_parameters, cfg.calibrated_vectors)
print("Parameters loaded:", len(params.table))

Parameters loaded: 125


### Fiscal calendar

- **Year end:** the Saturday closest to January 31, so every fiscal year starts on a Sunday and ends on a Saturday. Years are named by the calendar year they end in.
- **Quarters and months:** each quarter has 13 weeks, split into fiscal months of 4, 5 and 4 weeks. Every month therefore contains whole weeks, so a month never splits a week and weekday patterns compare cleanly year to year.
- **53-week years:** when the closest Saturday falls after January 31 often enough, a year gains a 53rd week. The extra week is added to the last fiscal month.
- **Range:** FY2025 to FY2030, covering the actual period (from February 2024) and the long-range plan horizon.

In [2]:
dim_date = calendar.build_dim_date(cfg.first_fiscal_year, cfg.last_fiscal_year)
years = dim_date.groupby("fiscal_year").agg(start=("date", "min"), end=("date", "max"),
                                            weeks=("fiscal_week", "max"), days=("date", "size"))
years

,start,end,weeks,days
fiscal_year,,,,
2025,2024-02-04,2025-02-01,52,364
2026,2025-02-02,2026-01-31,52,364
2027,2026-02-01,2027-01-30,52,364
2028,2027-01-31,2028-01-29,52,364
2029,2028-01-30,2029-02-03,53,371
2030,2029-02-04,2030-02-02,52,364


**How to read the table:** each fiscal year starts on a Sunday and ends on a Saturday. FY2027, the current year in the story, runs from 1 February 2026 to 30 January 2027, and its third quarter closed on 31 October 2026. FY2029 is the only 53-week year in the range (371 days), which is why growth targets for FY2029 must adjust for the extra week.

In [3]:
fy27 = dim_date[dim_date.fiscal_year == 2027]
fy27.groupby("fiscal_quarter").agg(start=("date", "min"), end=("date", "max"), weeks=("fiscal_week", "nunique"))

,start,end,weeks
fiscal_quarter,,,
1,2026-02-01,2026-05-02,13
2,2026-05-03,2026-08-01,13
3,2026-08-02,2026-10-31,13
4,2026-11-01,2027-01-30,13


### Holidays and store closures

Store closures follow provincial retail closing laws, recorded in `config/structure/closure_rules.csv` and backed by the `GOV-LAW` rows of the benchmark register. Each holiday gets one row per province with a trading status: closed, reduced hours or open.

- **Ontario:** all stores close on the nine days in the Retail Business Holidays Act, including stores in the City of Toronto, where the Act does not apply; closing them is company policy. When July 1 is a Sunday, Ontario's Canada Day closure moves to the Monday.
- **Reduced hours:** Christmas Eve and New Year's Eve close at 5 p.m. (required in Quebec, company policy elsewhere); Quebec stores open at 1 p.m. on Boxing Day.
- **Sundays in New Brunswick:** the Days of Rest Act closes retail on Sundays unless exempted; Kettlerock's New Brunswick stores are assumed to open under local exemptions (CAL-02), so Sundays are not listed as closures.

In [4]:
dim_holiday = calendar.build_dim_holiday(cfg.structure_dir, cfg.first_fiscal_year, cfg.last_fiscal_year)
h26 = dim_holiday[dim_holiday.calendar_year == 2026]
h26.pivot_table(index=["date", "holiday_key"], columns="province", values="trading_status", aggfunc="first")

,province,NB,NS,ON,QC
date,holiday_key,,,,
2026-01-01,new_years_day,closed,closed,closed,closed
2026-02-16,family_day,closed,closed,closed,open
2026-04-03,good_friday,closed,closed,closed,open
2026-04-05,easter_sunday,open,closed,closed,closed
2026-05-18,victoria_day,closed,open,closed,open
2026-06-24,saint_jean_baptiste,open,open,open,closed
2026-07-01,canada_day,closed,closed,closed,closed
2026-08-03,new_brunswick_day,closed,open,open,open
2026-09-07,labour_day,closed,closed,closed,closed


**How to read the table:** each row is one holiday in 2026 and each column a province. Christmas Day is closed everywhere; Boxing Day is open in Ontario, opens at 1 p.m. in Quebec, and is closed in Nova Scotia and New Brunswick. Quebec stays open on Good Friday, Victoria Day and Thanksgiving but closes on Saint-Jean-Baptiste Day; Remembrance Day closes stores only in Nova Scotia and New Brunswick.

In [5]:
closed_days = (dim_holiday[dim_holiday.trading_status == "closed"]
               .groupby(["calendar_year", "province"]).size().unstack())
closed_days.loc[2025:2027]

province,NB,NS,ON,QC
calendar_year,,,,
2025,11,10,9,6
2026,11,10,9,6
2027,11,10,9,6


**How to read the table:** the number of legally closed days per province per year. Nova Scotia and New Brunswick close on more days than Ontario and Quebec, so Atlantic stores lose more trading days, which matters when comparing their sales with the rest of the chain.

### Stores and markets

The store list in `config/structure/stores_master.csv` is authored to match the store timeline and market layout in the specification: 30 stores at the start of FY2025, 40 at the end of Q3 FY2027, and two Ontario openings planned for Q4.

- **Identifiers:** stores are numbered `KVS001` to `KVS042` and never renumbered.
- **Formats:** Atlantic stores use the 6,000 sq ft compact format; all others are 9,000 sq ft.
- **Leases:** a 10-year base term from opening, renewed in 5-year steps; the table holds the term current at the Q3 FY2027 close. Stores are remodelled when their lease is renewed (nine remodels since February 2024).
- **Occupancy:** each store's occupancy cost per square foot is its regional rate (SE-11a to SE-11d) times a location multiplier (SE-11e to SE-11g).

In [6]:
dim_store = structure.build_dim_store(cfg.structure_dir, params)
fy_bounds = dim_date.groupby("fiscal_year").date.agg(["min", "max"])
opened = dim_store[dim_store.status == "open"].copy()
opened["opened_in_fy"] = opened.open_date.map(
    lambda d: next((fy for fy, r in fy_bounds.iterrows() if r["min"] <= d <= r["max"]), "before FY2025"))
pd.crosstab(opened.opened_in_fy, opened.province, margins=True, margins_name="Total")

province,NB,NS,ON,QC,Total
opened_in_fy,,,,,
2025,0,1,2,1,4
2026,1,0,1,1,3
2027,1,2,0,0,3
before FY2025,0,0,23,7,30
Total,2,3,26,9,40


**How to read the table:** 30 stores were open before FY2025 (23 in Ontario, 7 in Quebec). Four opened in FY2025, three in FY2026 and three in FY2027 up to the Q3 close, all three of those in Atlantic Canada, giving 40 stores at the end of Q3 FY2027.

In [7]:
open_q3 = dim_store[(dim_store.status == "open") & (dim_store.open_date <= "2026-10-31")]
markets = open_q3.groupby(["region", "market_id"]).size().rename("stores")
markets.groupby(level=0).agg(["count", "sum"]).rename(columns={"count": "markets", "sum": "stores"})

,markets,stores
region,,
Atlantic Canada,4,5
Greater Toronto Area,5,14
Montreal area,4,9
Southwestern Ontario,8,12


**How to read the table:** 40 stores in 21 trade areas. Each market holds one to three stores, and a local event such as a competitor opening affects every store in its market.

In [8]:
mature = params.value("SE-03")
sales = open_q3.format.map({"standard": mature, "compact": mature * params.value("SE-04")})
occupancy = pd.DataFrame({"occupancy": open_q3.annual_occupancy_cost.astype(float), "mature_sales": sales,
                          "region": open_q3.region}).groupby("region").sum()
occupancy.loc["Company"] = occupancy.sum()
occupancy["occupancy_pct_of_mature_sales"] = (occupancy.occupancy / occupancy.mature_sales * 100).round(1)
occupancy

,occupancy,mature_sales,occupancy_pct_of_mature_sales
region,,,
Atlantic Canada,641700.0,11062500.0,5.8
Greater Toronto Area,4306500.0,41300000.0,10.4
Montreal area,2343600.0,26550000.0,8.8
Southwestern Ontario,2527200.0,35400000.0,7.1
Company,9819000.0,114312500.0,8.6


**How to read the table:** rent rates apply per selling square foot, and each store's rate is its regional rate multiplied by its location multiplier (a GTA urban store pays $33 × 1.10 = $36.30). At mature sales, occupancy comes to about 8.6% of sales for the company, inside the 8–9% range set for SE-11. It is highest in the Greater Toronto Area (about 10%) and lowest in Atlantic Canada (about 6%), so GTA stores need higher sales to earn the same contribution.

In [9]:
ends = pd.to_datetime(dim_store.lease_term_end)
expiring = dim_store[(ends >= "2027-01-31") & (ends <= "2029-02-03")]
expiring[["store_id", "city", "open_date", "lease_term_end", "lease_renewals_taken"]]

,store_id,city,open_date,lease_term_end,lease_renewals_taken
1,KVS002,Brampton,2013-06-08,2028-06-08,1
2,KVS003,Brampton,2018-05-12,2028-05-12,0
4,KVS005,Scarborough,2012-10-13,2027-10-13,1
7,KVS008,Mississauga,2017-04-08,2027-04-08,0
17,KVS018,Kitchener,2018-09-08,2028-09-08,0
18,KVS019,Hamilton,2012-04-21,2027-04-21,1
20,KVS021,Windsor,2017-10-14,2027-10-14,0
22,KVS023,Woodstock,2018-11-10,2028-11-10,0
29,KVS030,Montreal,2013-09-14,2028-09-14,1


**How to read the table:** nine store leases expire in FY2028 or FY2029. These are the renewal, relocation or closure decisions that the store portfolio review later in the analysis has to make.

### Products

The catalogue in `config/structure/products_master.csv` holds 180 items in 11 departments across three categories. Item attributes are deterministic: price points cycle through each department's allowed set, and item margins are spread evenly within three points of the category margin, so each category averages exactly its calibrated product margin (MP-05 to MP-07).

- **Imports:** a set share of items in each department is imported and priced in US dollars; landed cost adds inbound freight (MP-14) and duty (MP-15), with higher duty on toys and seasonal goods.
- **Opening cost:** the first row of `fact_cost_history` records each item's supplier cost from the start of FY2025; later cost changes come from the world draw.

In [10]:
dim_product = structure.build_dim_product(cfg.structure_dir)
p = dim_product.astype({"price_point": float, "product_margin_at_ticket": float, "import_flag": int})
p.groupby("category").agg(items=("product_id", "size"), departments=("department", "nunique"),
                          average_price=("price_point", "mean"), product_margin=("product_margin_at_ticket", "mean"),
                          imported_share_of_items=("import_flag", "mean")).round(3)

,items,departments,average_price,product_margin,imported_share_of_items
category,,,,,
Consumables,72,3,2.253,0.37,0.167
General merchandise,66,3,3.144,0.49,0.758
Seasonal,42,5,3.131,0.52,0.881


**How to read the table:** product margins average 37%, 49% and 52%, the calibrated values. Consumables are the cheapest items (average price $2.25) and mostly domestic; general merchandise and seasonal goods are mostly imported. Weighted by the 45/40/15 sales mix, the average item price is about $2.67, consistent with an $11.50 average ticket of about 4.3 items ($2.674). Imports make up about 47% of cost of goods at landed cost, inside the 45–52% range for MP-13 and in line with Dollarama's 46–48% of purchase volume.

### Writing the canonical files

Each table is written once as a canonical CSV in `output/csv`, with fixed formatting so the same inputs always produce the same bytes. `manifest.csv` records each file's SHA-256 hash and row count; PostgreSQL loads and published Parquet files are checked against it.

In [11]:
from kettlerock_engine.build import build_structure
from kettlerock_engine.outputs import write_table, write_manifest

tables = build_structure(cfg)
entries = [write_table(df, name, cfg.output_dir) for name, df in tables.items()]
write_manifest(entries, cfg.output_dir)
pd.DataFrame(entries).assign(sha256=lambda d: d.sha256.str[:12])

,file,sha256,rows
0,dim_date.csv,a24c02b4edf1,2191
1,dim_holiday.csv,bcecca3a50b2,360
2,dim_store.csv,5f9b1cdc4e82,42
3,dim_market.csv,8de6d0f88535,22
4,dim_product.csv,11f3ec8895ee,180
5,fact_cost_history.csv,ec46e52df6f2,180
6,dim_account.csv,e4b77c199636,20
7,dim_scenario.csv,ad23c8658744,10
8,dim_debt_facility.csv,2f45c05edc80,2


**How to read the table:** nine structure tables, from 2 rows (the debt facilities) to 2,191 (one row per day from FY2025 to FY2030). The short hash identifies each file's exact contents; rebuilding gives the same hashes.

The tests in `tests/` check the same facts independently (fiscal year bounds, the 4-5-4 pattern, closure statuses, the store timeline, product margins and byte-identical rebuilds). Run them from the `02_engine` folder with `python -m pytest`.